# FlowThought PoC — A100 Definitive Experiment

**Flow Matching for Reasoning in LLM Hidden State Space**

Uses **DeepSeek-R1-Distill-Qwen-7B** (an actual RLVR-trained reasoning model) on full GSM8K.
Requires Colab A100 runtime (40/80GB VRAM).

**Goal: definitive go/no-go.** If FM-generated hidden states can't beat random injection
on a real reasoning model with full data, the approach is dead.

Pipeline:
1. **Phase 1** — Extract hidden states from DeepSeek-R1-Distill-Qwen-7B on full GSM8K
2. **Phase 2** — Train a CFM velocity field (VelocityMLP) to map noise → CoT hidden states
3. **Phase 3a** — Probe evaluation: correlation of FM states with true answers
4. **Phase 3b** — Coconut-style injection: inject FM states into residual stream, measure accuracy

In [ ]:
# Cell 1: Install dependencies (torch is pre-installed on Colab)
!pip install -q transformers==4.47.0 datasets==3.2.0 accelerate==1.2.1 matplotlib==3.9.3
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "Flash Attention 2 not available, falling back to SDPA"

In [ ]:
# Cell 2: Imports + Config (A100 + DeepSeek-R1-Distill-Qwen-7B)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import re
import gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

# Detect Flash Attention 2 availability
try:
    import flash_attn
    ATTN_IMPL = "flash_attention_2"
    print("Flash Attention 2: available")
except ImportError:
    ATTN_IMPL = "sdpa"
    print("Flash Attention 2: not available, using SDPA")

CONFIG = {
    "model_name": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "hidden_dim": 3584,          # Qwen2-7B hidden size
    "n_train": 7473,             # full GSM8K train
    "n_test": 1319,              # full GSM8K test
    "batch_size_extract": 128,   # A100 can handle larger batches (was 64)
    "batch_size_extract_long": 16, # for long texts (problem+CoT) — A100 safe (was 8)
    "max_length": 1024,          # reasoning models produce longer CoT
    # FM training
    "fm_epochs": 1000,           # more epochs for 3584-dim space (was 500)
    "fm_batch_size": 4096,       # large batch for variance reduction
    "fm_lr": 1e-4,               # base LR (with warmup)
    "fm_warmup_epochs": 50,      # linear warmup for stability
    "fm_hidden": 4096,           # wider for 3584-dim output (was 2048)
    "fm_layers": 8,              # deeper (was 6)
    "grad_clip": 1.0,
    "fm_checkpoint_every": 50,
    "time_embed_dim": 128,       # larger time embedding
    # ODE
    "ode_steps": 30,             # RK4 steps
    # Probe
    "probe_epochs": 200,
    "probe_lr": 5e-4,
    "probe_hidden": 512,
    # Paths
    "cache_dir": "flowthought_cache_r1_7b",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {mem_gb:.1f} GB")
    assert mem_gb > 30, f"Need A100 (40+ GB), got {mem_gb:.1f} GB. Change runtime to A100!"

In [ ]:
# Cell 3: Test framework
@dataclass
class TestResults:
    passed: int = 0
    failed: int = 0
    warned: int = 0
    details: List[str] = field(default_factory=list)

    def check(self, condition: bool, name: str):
        if condition:
            self.passed += 1
            self.details.append(f"  PASS: {name}")
        else:
            self.failed += 1
            self.details.append(f"  FAIL: {name}")

    def warn(self, condition: bool, name: str):
        """Soft check — logs WARN instead of FAIL, does not block execution."""
        if condition:
            self.passed += 1
            self.details.append(f"  PASS: {name}")
        else:
            self.warned += 1
            self.details.append(f"  WARN: {name}")

    def summary(self, section: str):
        total = self.passed + self.failed + self.warned
        status = "ALL PASSED" if self.failed == 0 and self.warned == 0 else ""
        if self.failed > 0:
            status = f"{self.failed} FAILED"
        elif self.warned > 0:
            status = f"PASSED with {self.warned} warning(s)"
        print(f"\n[{section}] {self.passed}/{total} tests passed — {status}")
        for d in self.details:
            print(d)
        assert self.failed == 0, f"{self.failed} tests failed in {section}"

# Quick self-test
t = TestResults()
t.check(True, "framework works")
t.summary("Self-test")

In [ ]:
# Cell 4: Data loading — GSM8K + answer parsing
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main")
print(f"Train: {len(ds['train'])}, Test: {len(ds['test'])}")

def parse_answer(answer_str: str) -> Optional[float]:
    """Extract the numeric answer after #### from GSM8K format."""
    match = re.search(r"####\s*([\-\d,\.]+)", answer_str)
    if match:
        return float(match.group(1).replace(",", ""))
    return None

def get_cot_and_answer(example):
    """Split GSM8K answer field into CoT reasoning and numeric answer."""
    text = example["answer"]
    parts = text.split("####")
    cot = parts[0].strip()
    ans = parse_answer(text)
    return cot, ans

# Tests
t = TestResults()
t.check(parse_answer("blah blah\n#### 42") == 42.0, "parse simple")
t.check(parse_answer("stuff\n#### 1,234") == 1234.0, "parse comma")
t.check(parse_answer("stuff\n#### -5") == -5.0, "parse negative")
t.check(parse_answer("no answer here") is None, "parse missing")

cot, ans = get_cot_and_answer(ds["train"][0])
t.check(isinstance(cot, str) and len(cot) > 10, "cot is string")
t.check(isinstance(ans, float), "answer is float")
t.summary("Data Loading")

print(f"\nExample question: {ds['train'][0]['question'][:100]}...")
print(f"Answer: {ans}")

In [ ]:
# Cell 5: Model loading — DeepSeek-R1-Distill-Qwen-7B on A100 (FA2 if available)
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.bfloat16,
    attn_implementation=ATTN_IMPL,
).to(device)
model.eval()

# Register forward hook on the final layernorm (after all transformer layers)
_last_hidden = {}
def _hook_fn(module, input, output):
    _last_hidden["val"] = output

_hook_handle = model.model.norm.register_forward_hook(_hook_fn)

# Test
t = TestResults()
test_input = tokenizer("Hello", return_tensors="pt").to(device)
with torch.no_grad():
    _ = model(**test_input)
last_hidden = _last_hidden["val"]
t.check(last_hidden.shape[-1] == CONFIG["hidden_dim"], f"hidden dim = {last_hidden.shape[-1]}")
t.check(not torch.isnan(last_hidden).any(), "no NaN in hidden states")
t.summary("Model Loading")
print(f"Model loaded: {CONFIG['model_name']} (bf16 + {ATTN_IMPL})")
if device.type == "cuda":
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell 6: Hidden state extraction functions (hook-based, no output_hidden_states overhead)

@torch.no_grad()
def extract_hidden_states(texts: List[str], batch_size: int = 32) -> torch.Tensor:
    """Extract last-layer, last-token hidden states via forward hook.
    Returns: (N, hidden_dim) float32 tensor on CPU.
    """
    all_states = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    n_truncated = 0
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True,
            max_length=CONFIG["max_length"],
        ).to(device)
        # Count truncated examples
        for j in range(len(batch)):
            if inputs["attention_mask"][j].sum() == CONFIG["max_length"]:
                n_truncated += 1
        # Forward pass — hook captures last layer output
        _ = model(**inputs)
        hidden = _last_hidden["val"]  # (B, seq_len, hidden_dim)
        # With left-padding, last token is always the last position
        last_token_states = hidden[:, -1, :]  # (B, hidden_dim)
        all_states.append(last_token_states.float().cpu())
        if (i // batch_size) % 25 == 0:
            print(f"    batch {i // batch_size + 1}/{n_batches}")
    if n_truncated > 0:
        print(f"    WARNING: {n_truncated}/{len(texts)} examples truncated at max_length={CONFIG['max_length']}")
    return torch.cat(all_states, dim=0)

# Tests
t = TestResults()
test_texts = ["What is 2+2?", "The answer to 2+2 is 4 because addition."]
test_states = extract_hidden_states(test_texts, batch_size=2)
t.check(test_states.shape == (2, CONFIG["hidden_dim"]), f"shape: {test_states.shape}")
t.check(not torch.isnan(test_states).any(), "no NaN")
t.check(not torch.isinf(test_states).any(), "no Inf")
cos_sim = F.cosine_similarity(test_states[0:1], test_states[1:2]).item()
t.check(cos_sim < 0.99, f"states are distinct (cos_sim={cos_sim:.4f})")
t.summary("Hidden State Extraction")

In [ ]:
# Cell 7: Phase 1 — Run extraction on GSM8K (with caching) + free LLM after

cache_dir = Path(CONFIG["cache_dir"])
cache_dir.mkdir(exist_ok=True)

def run_extraction(split, n_examples):
    """Extract condition (problem) and target (problem+CoT) hidden states."""
    cache_file = cache_dir / f"{split}_{n_examples}.pt"
    if cache_file.exists():
        print(f"Loading cached {split} data from {cache_file}")
        return torch.load(cache_file, weights_only=False)

    data = ds[split].select(range(min(n_examples, len(ds[split]))))
    problems = []
    problems_with_cot = []
    answers = []
    skipped = 0

    for ex in data:
        q = ex["question"]
        cot, ans = get_cot_and_answer(ex)
        if ans is None:
            skipped += 1
            continue
        problems.append(q)
        problems_with_cot.append(f"{q}\n{cot}")
        answers.append(ans)

    if skipped > 0:
        print(f"  Skipped {skipped} examples with unparseable answers")
    print(f"Extracting {len(problems)} {split} examples...")

    print("  Extracting condition (problem-only) hidden states...")
    h_cond = extract_hidden_states(problems, CONFIG["batch_size_extract"])

    print("  Extracting target (problem+CoT) hidden states (smaller batches for long seqs)...")
    h_target = extract_hidden_states(problems_with_cot, CONFIG["batch_size_extract_long"])

    answers_tensor = torch.tensor(answers, dtype=torch.float32)

    result = {"h_cond": h_cond, "h_target": h_target, "answers": answers_tensor}
    torch.save(result, cache_file)
    print(f"  Saved to {cache_file}")
    return result

train_data = run_extraction("train", CONFIG["n_train"])
test_data = run_extraction("test", CONFIG["n_test"])

# Free LLM from GPU — it's not needed for Phase 2/3
_hook_handle.remove()
del model, tokenizer, _last_hidden
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
    print(f"\nLLM freed. GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("\nLLM freed from memory.")

# z-score normalization (fit on train)
h_mean = train_data["h_target"].mean(dim=0)
h_std = train_data["h_target"].std(dim=0).clamp(min=1e-6)
a_mean = train_data["answers"].mean()
a_std = train_data["answers"].std().clamp(min=1e-6)

def normalize_h(h):
    return (h - h_mean) / h_std

def normalize_a(a):
    return (a - a_mean) / a_std

def denormalize_a(a):
    return a * a_std + a_mean

# Tests
t = TestResults()
for name, d in [("train", train_data), ("test", test_data)]:
    t.check(d["h_cond"].shape[1] == CONFIG["hidden_dim"], f"{name} cond dim")
    t.check(d["h_target"].shape[1] == CONFIG["hidden_dim"], f"{name} target dim")
    t.check(not torch.isnan(d["h_cond"]).any(), f"{name} cond no NaN")
    t.check(not torch.isnan(d["h_target"]).any(), f"{name} target no NaN")
    cos = F.cosine_similarity(d["h_cond"], d["h_target"]).mean().item()
    t.check(cos < 0.99, f"{name} cond != target (mean cos={cos:.4f})")

t.check(train_data["h_cond"].shape[0] >= CONFIG["n_train"] * 0.9, "enough train examples")
t.check(test_data["h_cond"].shape[0] >= CONFIG["n_test"] * 0.9, "enough test examples")
t.summary("Phase 1: Extraction")

print(f"\nTrain: {train_data['h_cond'].shape[0]} examples")
print(f"Test: {test_data['h_cond'].shape[0]} examples")
print(f"Hidden dim: {CONFIG['hidden_dim']}")

In [ ]:
# Cell 8: VelocityMLP with residual blocks + AdaLN conditioning + zero-init output
import math

class TimestepEmbedding(nn.Module):
    """DDPM-style sinusoidal timestep embedding (Ho et al. 2020 / Vaswani et al. 2017)."""
    def __init__(self, embed_dim=64):
        super().__init__()
        half = embed_dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, dtype=torch.float32) / max(half - 1, 1))
        self.register_buffer("freqs", freqs)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.SiLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, t):
        freq_t = t * self.freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(freq_t), torch.cos(freq_t)], dim=-1)
        return self.mlp(emb)


class AdaLN(nn.Module):
    """Adaptive Layer Normalization — modulates scale/shift from conditioning."""
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.proj = nn.Linear(cond_dim, dim * 2)

    def forward(self, x, cond):
        scale, shift = self.proj(cond).chunk(2, dim=-1)
        return self.norm(x) * (1 + scale) + shift


class ResidualBlock(nn.Module):
    """Residual block with AdaLN conditioning on time+condition embedding."""
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.linear1 = nn.Linear(dim, dim)
        self.adaln = AdaLN(dim, cond_dim)
        self.linear2 = nn.Linear(dim, dim)
        # Zero-init second linear so block starts as identity
        nn.init.zeros_(self.linear2.weight)
        nn.init.zeros_(self.linear2.bias)

    def forward(self, x, cond):
        h = F.silu(self.linear1(x))
        h = self.adaln(h, cond)
        h = self.linear2(h)
        return x + h


class VelocityMLP(nn.Module):
    """MLP velocity field for CFM with residual blocks + AdaLN conditioning.

    Architecture:
    - Input projection: [h_t, t_emb, c] → mlp_hidden
    - Condition projection: [t_emb, c] → cond_dim (shared across blocks)
    - N residual blocks with AdaLN
    - Zero-init output projection → hidden_dim
    """
    def __init__(self, hidden_dim, mlp_hidden, n_layers, time_embed_dim=64):
        super().__init__()
        self.time_embed = TimestepEmbedding(time_embed_dim)
        input_dim = hidden_dim * 2 + time_embed_dim
        cond_dim = hidden_dim + time_embed_dim  # [t_emb, c] for AdaLN

        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, mlp_hidden),
            nn.SiLU(),
        )

        # Residual blocks with AdaLN
        self.blocks = nn.ModuleList([
            ResidualBlock(mlp_hidden, cond_dim) for _ in range(n_layers)
        ])

        # Output projection (zero-init for stable start)
        self.output_proj = nn.Linear(mlp_hidden, hidden_dim)
        nn.init.zeros_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)

    def forward(self, h_t, t, c):
        t_emb = self.time_embed(t)
        x = torch.cat([h_t, t_emb, c], dim=-1)
        cond = torch.cat([t_emb, c], dim=-1)  # conditioning for AdaLN

        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x, cond)
        return self.output_proj(x)

# Tests
t = TestResults()
test_mlp = VelocityMLP(
    CONFIG["hidden_dim"], CONFIG["fm_hidden"], CONFIG["fm_layers"],
    time_embed_dim=CONFIG["time_embed_dim"],
).to(device)
B = 4
h_t = torch.randn(B, CONFIG["hidden_dim"], device=device)
t_in = torch.rand(B, 1, device=device)
c = torch.randn(B, CONFIG["hidden_dim"], device=device)
out = test_mlp(h_t, t_in, c)
t.check(out.shape == (B, CONFIG["hidden_dim"]), f"output shape: {out.shape}")
t.check(not torch.isnan(out).any(), "no NaN")
t.check(not torch.isinf(out).any(), "no Inf")
# Zero-init means initial output should be near zero
t.check(out.abs().mean().item() < 1.0, f"zero-init output near zero: {out.abs().mean().item():.4f}")
n_params = sum(p.numel() for p in test_mlp.parameters())
t.check(n_params > 0, f"has {n_params:,} parameters")
t.summary("VelocityMLP (Residual + AdaLN)")
del test_mlp

In [ ]:
# Cell 9: CFM loss function (logit-normal time sampling + t-clamping)

def cfm_loss(velocity_net, h_target, h_cond):
    """
    Conditional Flow Matching loss with logit-normal time sampling.
    - Source: h_0 ~ N(0, I)
    - Target: h_1 = normalized CoT hidden states
    - Condition: c = normalized problem hidden states
    - Linear interpolation: h_t = (1-t)*h_0 + t*h_1
    - True velocity: u_t = h_1 - h_0
    - Loss: MSE(v_theta(h_t, t, c), u_t)
    """
    B = h_target.shape[0]
    h_0 = torch.randn_like(h_target)

    # Logit-normal time sampling, clamped away from boundaries
    t = torch.sigmoid(torch.randn(B, 1, device=h_target.device))
    t = t.clamp(min=0.001, max=0.999)

    # Linear interpolation
    h_t = (1 - t) * h_0 + t * h_target

    # True velocity (linear path)
    u_t = h_target - h_0

    # Predicted velocity
    v_pred = velocity_net(h_t, t, h_cond)

    return F.mse_loss(v_pred, u_t)

# Test: loss decreases over a few steps
t = TestResults()
test_net = VelocityMLP(CONFIG["hidden_dim"], 256, 2).to(device)
test_opt = torch.optim.Adam(test_net.parameters(), lr=1e-3)

h_tgt_small = normalize_h(train_data["h_target"][:32]).to(device)
h_cnd_small = normalize_h(train_data["h_cond"][:32]).to(device)

losses = []
for step in range(20):
    test_opt.zero_grad()
    loss = cfm_loss(test_net, h_tgt_small, h_cnd_small)
    loss.backward()
    test_opt.step()
    losses.append(loss.item())

t.check(losses[-1] < losses[0], f"loss decreased: {losses[0]:.4f} -> {losses[-1]:.4f}")
t.check(not np.isnan(losses[-1]), "loss not NaN")
t.summary("CFM Loss")
del test_net, test_opt

In [ ]:
# Cell 10: Phase 2 — Train FM (AMP + EMA + warmup + checkpointing)
import copy

velocity_net = VelocityMLP(
    CONFIG["hidden_dim"], CONFIG["fm_hidden"], CONFIG["fm_layers"],
    time_embed_dim=CONFIG["time_embed_dim"],
).to(device)

# EMA copy for stable inference
ema_net = copy.deepcopy(velocity_net)
ema_decay = 0.999

warmup_epochs = CONFIG["fm_warmup_epochs"]
cosine_epochs = CONFIG["fm_epochs"] - warmup_epochs  # scheduler only covers post-warmup

optimizer = torch.optim.AdamW(velocity_net.parameters(), lr=CONFIG["fm_lr"], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cosine_epochs)
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

# Prepare normalized data — all on GPU
h_target_train = normalize_h(train_data["h_target"]).to(device)
h_cond_train = normalize_h(train_data["h_cond"]).to(device)

n_train = h_target_train.shape[0]
loss_history = []
checkpoint_dir = cache_dir / "fm_checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

# Check for existing checkpoint to resume from
start_epoch = 0
latest_ckpt = checkpoint_dir / "latest.pt"
if latest_ckpt.exists():
    ckpt = torch.load(latest_ckpt, weights_only=False)
    try:
        velocity_net.load_state_dict(ckpt["model"])
        ema_net.load_state_dict(ckpt["ema"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        if "scaler" in ckpt:
            scaler.load_state_dict(ckpt["scaler"])
        loss_history = ckpt["loss_history"]
        start_epoch = ckpt["epoch"] + 1
        print(f"Resumed from checkpoint at epoch {start_epoch}")
    except RuntimeError as e:
        print(f"Checkpoint incompatible, training from scratch: {e}")
        latest_ckpt.unlink()
        loss_history = []
        start_epoch = 0
        optimizer = torch.optim.AdamW(velocity_net.parameters(), lr=CONFIG["fm_lr"], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cosine_epochs)
        scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

print(f"Training FM: epochs {start_epoch}-{CONFIG['fm_epochs']}, {n_train} examples")
print(f"  batch_size={CONFIG['fm_batch_size']}, warmup={warmup_epochs} epochs, cosine={cosine_epochs} epochs")
print(f"  VelocityMLP: {CONFIG['fm_layers']} ResidualBlocks x {CONFIG['fm_hidden']} hidden + AdaLN")
print(f"  Params: {sum(p.numel() for p in velocity_net.parameters()):,}")
if device.type == "cuda":
    print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

for epoch in range(start_epoch, CONFIG["fm_epochs"]):
    # Linear warmup: ramp LR from 0 to fm_lr over warmup_epochs
    if epoch < warmup_epochs:
        warmup_factor = (epoch + 1) / warmup_epochs
        for pg in optimizer.param_groups:
            pg["lr"] = CONFIG["fm_lr"] * warmup_factor

    perm = torch.randperm(n_train, device=device)
    epoch_losses = []

    for i in range(0, n_train, CONFIG["fm_batch_size"]):
        idx = perm[i:i + CONFIG["fm_batch_size"]]
        h_tgt = h_target_train[idx]
        h_cnd = h_cond_train[idx]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            loss = cfm_loss(velocity_net, h_tgt, h_cnd)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(velocity_net.parameters(), CONFIG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        epoch_losses.append(loss.item())

    # EMA update
    with torch.no_grad():
        for p_ema, p_model in zip(ema_net.parameters(), velocity_net.parameters()):
            p_ema.mul_(ema_decay).add_(p_model, alpha=1 - ema_decay)

    # Cosine schedule only after warmup completes
    if epoch >= warmup_epochs:
        scheduler.step()

    avg_loss = np.mean(epoch_losses)
    loss_history.append(avg_loss)

    if (epoch + 1) % 100 == 0 or epoch == start_epoch:
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch+1:4d}/{CONFIG['fm_epochs']} | Loss: {avg_loss:.6f} | LR: {lr_now:.2e}")

    # Checkpoint
    if (epoch + 1) % CONFIG["fm_checkpoint_every"] == 0:
        torch.save({
            "epoch": epoch,
            "model": velocity_net.state_dict(),
            "ema": ema_net.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "loss_history": loss_history,
        }, latest_ckpt)

# Plot
plt.figure(figsize=(8, 4))
plt.semilogy(loss_history)
plt.axvline(x=warmup_epochs, color='r', linestyle='--', alpha=0.5, label=f'warmup end ({warmup_epochs})')
plt.xlabel("Epoch")
plt.ylabel("Loss (log)")
plt.title("FM Training Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Tests
t = TestResults()
pct_decrease = (loss_history[0] - loss_history[-1]) / loss_history[0] * 100
t.check(loss_history[-1] < loss_history[0], f"loss decreased: {loss_history[0]:.4f} -> {loss_history[-1]:.4f} ({pct_decrease:.0f}%)")
t.warn(pct_decrease > 25, f"loss decreased by >25% (got {pct_decrease:.0f}%)")
t.summary("Phase 2: FM Training")

In [ ]:
# Cell 11: ODE integration (RK4 — same quality as Euler-50 in ~20 steps)

@torch.no_grad()
def rk4_integrate(velocity_net, h_cond, n_steps=20, device=None):
    """
    RK4-integrate the learned velocity field from t=0 to t=1.
    h_cond: (B, hidden_dim) — normalized condition vectors
    Returns: (B, hidden_dim) — generated hidden states at t=1
    """
    if device is None:
        device = h_cond.device
    B, D = h_cond.shape
    dt = 1.0 / n_steps
    h_t = torch.randn(B, D, device=device)

    for step in range(n_steps):
        t_val = step * dt
        t1 = torch.full((B, 1), t_val, device=device)
        t2 = torch.full((B, 1), t_val + dt / 2, device=device)
        t3 = torch.full((B, 1), t_val + dt, device=device)

        k1 = velocity_net(h_t, t1, h_cond)
        k2 = velocity_net(h_t + k1 * dt / 2, t2, h_cond)
        k3 = velocity_net(h_t + k2 * dt / 2, t2, h_cond)
        k4 = velocity_net(h_t + k3 * dt, t3, h_cond)

        h_t = h_t + (k1 + 2 * k2 + 2 * k3 + k4) * dt / 6

    return h_t

# Tests — use EMA net for inference (more stable)
t = TestResults()
test_cond = normalize_h(test_data["h_cond"][:16]).to(device)
generated = rk4_integrate(ema_net, test_cond, CONFIG["ode_steps"])
t.check(generated.shape == (16, CONFIG["hidden_dim"]), f"shape: {generated.shape}")
t.check(not torch.isnan(generated).any(), "no NaN")
t.check(not torch.isinf(generated).any(), "no Inf")
pairwise_cos = F.cosine_similarity(generated[:-1], generated[1:]).mean().item()
t.check(pairwise_cos < 0.99, f"not collapsed (avg pairwise cos={pairwise_cos:.4f})")
t.summary("RK4 ODE Integration")

In [ ]:
# Cell 12: Answer probe — MLP trained to predict answer from hidden states

class AnswerProbe(nn.Module):
    def __init__(self, hidden_dim, probe_hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, probe_hidden),
            nn.ReLU(),
            nn.Linear(probe_hidden, probe_hidden),
            nn.ReLU(),
            nn.Linear(probe_hidden, 1),
        )

    def forward(self, h):
        return self.net(h).squeeze(-1)

# Train probe on real CoT hidden states
probe = AnswerProbe(CONFIG["hidden_dim"], CONFIG["probe_hidden"]).to(device)
probe_opt = torch.optim.Adam(probe.parameters(), lr=CONFIG["probe_lr"])

h_target_norm = normalize_h(train_data["h_target"]).to(device)
a_norm = normalize_a(train_data["answers"]).to(device)

probe_losses = []
for epoch in range(CONFIG["probe_epochs"]):
    perm = torch.randperm(h_target_norm.shape[0], device=device)
    epoch_loss = []
    for i in range(0, h_target_norm.shape[0], 128):
        idx = perm[i:i+128]
        pred = probe(h_target_norm[idx])
        loss = F.mse_loss(pred, a_norm[idx])
        probe_opt.zero_grad()
        loss.backward()
        probe_opt.step()
        epoch_loss.append(loss.item())
    probe_losses.append(np.mean(epoch_loss))

probe.eval()

# Test on train set
with torch.no_grad():
    train_pred = denormalize_a(probe(h_target_norm)).cpu()
    train_true = train_data["answers"]
    train_corr = np.corrcoef(train_pred.numpy(), train_true.numpy())[0, 1]

t = TestResults()
t.check(probe_losses[-1] < probe_losses[0], f"probe loss decreased: {probe_losses[0]:.4f} -> {probe_losses[-1]:.4f}")
t.check(not np.isnan(train_corr), f"train correlation is finite: {train_corr:.4f}")
t.warn(train_corr > 0.1, f"train correlation > 0.1 (got {train_corr:.4f}) — may need more data")
t.summary("Answer Probe")
print(f"Probe train correlation: {train_corr:.4f}")

In [ ]:
# Cell 13: Full evaluation — 4 baselines (using EMA net + RK4)

def evaluate_method(name, h_states, true_answers):
    """Evaluate hidden states via the answer probe."""
    with torch.no_grad():
        h_norm = normalize_h(h_states).to(device)
        pred_norm = probe(h_norm).cpu()
        pred = denormalize_a(pred_norm)
    true = true_answers.numpy()
    pred_np = pred.numpy()

    corr = np.corrcoef(pred_np, true)[0, 1] if len(true) > 1 else 0.0
    exact = np.mean(np.abs(np.round(pred_np) - true) < 0.5)
    mse = np.mean((pred_np - true) ** 2)

    return {"name": name, "correlation": corr, "exact_match": exact, "mse": mse}

# Prepare test data
test_answers = test_data["answers"]
n_test = test_data["h_cond"].shape[0]

# Method 1: No-CoT (condition/problem hidden states only)
r_nocot = evaluate_method("No-CoT", test_data["h_cond"], test_answers)

# Method 2: Real CoT (ground truth target hidden states)
r_real = evaluate_method("Real CoT", test_data["h_target"], test_answers)

# Method 3: Random (Gaussian noise)
r_random = evaluate_method("Random", torch.randn(n_test, CONFIG["hidden_dim"]) * h_std + h_mean, test_answers)

# Method 4: FlowThought (FM-generated hidden states via EMA + RK4)
h_cond_test_norm = normalize_h(test_data["h_cond"]).to(device)
h_flow = rk4_integrate(ema_net, h_cond_test_norm, CONFIG["ode_steps"])
h_flow_denorm = (h_flow.cpu() * h_std + h_mean)
r_flow = evaluate_method("FlowThought", h_flow_denorm, test_answers)

results = [r_nocot, r_real, r_random, r_flow]

# Diversity analysis
def diversity_score(h):
    """Mean pairwise cosine distance (1 - cos_sim) on a subsample."""
    h_sub = h[:min(200, len(h))]
    h_norm_vec = F.normalize(h_sub.float(), dim=-1)
    sim_matrix = h_norm_vec @ h_norm_vec.T
    mask = ~torch.eye(len(h_sub), dtype=torch.bool)
    return (1 - sim_matrix[mask].mean()).item()

div_real = diversity_score(test_data["h_target"])
div_flow = diversity_score(h_flow_denorm)
div_random = diversity_score(torch.randn(n_test, CONFIG["hidden_dim"]))

# Distribution match: mean/std distance between flow and real
real_mean = normalize_h(test_data["h_target"]).mean(dim=0)
flow_mean = h_flow.cpu().mean(dim=0)
mean_dist = (real_mean - flow_mean).norm().item()

real_std = normalize_h(test_data["h_target"]).std(dim=0)
flow_std = h_flow.cpu().std(dim=0)
std_dist = (real_std - flow_std).norm().item()

print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)
print(f"{'Method':<15} {'Correlation':>12} {'Exact Match':>12} {'MSE':>12}")
print("-"*51)
for r in results:
    print(f"{r['name']:<15} {r['correlation']:>12.4f} {r['exact_match']:>12.4f} {r['mse']:>12.1f}")

print(f"\nDiversity (cosine distance):")
print(f"  Real CoT: {div_real:.4f}")
print(f"  FlowThought: {div_flow:.4f}")
print(f"  Random: {div_random:.4f}")

print(f"\nDistribution match (FlowThought vs Real CoT):")
print(f"  Mean distance: {mean_dist:.4f}")
print(f"  Std distance: {std_dist:.4f}")

# Hard checks: no crashes, valid outputs
# Soft checks: quality metrics (may be weak with small data)
t = TestResults()
t.check(not np.isnan(r_flow["correlation"]), "FlowThought correlation not NaN")
t.check(not np.isnan(r_real["correlation"]), "Real CoT correlation not NaN")
t.check(div_flow > 0.001, f"FlowThought not fully collapsed (diversity={div_flow:.4f})")
t.warn(r_flow["correlation"] > r_random["correlation"],
       f"FlowThought > random ({r_flow['correlation']:.4f} vs {r_random['correlation']:.4f}) — scale up data if WARN")
t.warn(r_real["correlation"] > r_random["correlation"],
       f"Real CoT > random ({r_real['correlation']:.4f} vs {r_random['correlation']:.4f}) — probe may need more data")
t.summary("Full Evaluation")

In [ ]:
# Cell: Phase 3b — Reload LLM for injection (FA2 + multi-layer sweep)
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Reloading LLM for injection experiments...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.bfloat16,
    attn_implementation=ATTN_IMPL,
).to(device)
model.eval()

num_layers = len(model.model.layers)
# Test early-mid layers (research shows layers 8-13 range optimal for steering)
# For 28-layer model: try layers 7, 10, 14 (quarter, third, half)
inject_layer = num_layers // 4  # start with early-mid (layer ~7)
print(f"Model has {num_layers} layers, primary injection at layer {inject_layer}")
print(f"Will sweep layers: [{num_layers//4}, {num_layers//3}, {num_layers//2}]")

injection_state = {"active": False, "vector": None, "alpha": 0.1, "fired": False}
_current_hook_handle = None

def set_injection_layer(layer_idx):
    """Switch injection to a different layer."""
    global _current_hook_handle
    if _current_hook_handle is not None:
        _current_hook_handle.remove()
    _current_hook_handle = model.model.layers[layer_idx].register_forward_hook(injection_hook)
    return layer_idx

def injection_hook(module, input, output):
    """Add scaled FM state to residual stream at last token position.
    Fires ONCE per generation (prefill pass), then deactivates.
    """
    if not injection_state["active"] or injection_state["fired"]:
        return output
    hidden = output[0]
    vec = injection_state["vector"].to(hidden.device, hidden.dtype)
    hidden = hidden.clone()
    hidden[:, -1, :] = hidden[:, -1, :] + injection_state["alpha"] * vec
    injection_state["fired"] = True
    return (hidden,) + output[1:]

set_injection_layer(inject_layer)

# Tests
t = TestResults()
t.check(num_layers > 0, f"model has {num_layers} layers")
test_input = tokenizer("Hello", return_tensors="pt").to(device)
with torch.no_grad():
    _ = model(**test_input)
t.check(True, "forward pass works with hook registered")
t.summary("LLM Reload for Injection")
if device.type == "cuda":
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell: Generation helpers for injection evaluation

@torch.no_grad()
def generate_answer(question, inject_vector=None, alpha=0.1, max_new_tokens=50):
    """Generate an answer with optional hidden-state injection.
    
    Args:
        question: problem text
        inject_vector: (hidden_dim,) tensor — DENORMALIZED hidden state in LLM's native space
        alpha: scaling factor for injection
        max_new_tokens: generation budget
    Returns:
        generated text (answer portion only)
    """
    prompt = f"Question: {question}\nAnswer: The answer is"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CONFIG["max_length"]).to(device)
    
    if inject_vector is not None:
        injection_state["active"] = True
        injection_state["fired"] = False  # reset for new generation
        injection_state["vector"] = inject_vector.unsqueeze(0)  # (1, hidden_dim)
        injection_state["alpha"] = alpha
    else:
        injection_state["active"] = False
        injection_state["fired"] = False
    
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
    )
    
    injection_state["active"] = False
    generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated

def parse_generated_answer(text):
    """Extract first number from generated text."""
    match = re.search(r'[-+]?\d[\d,]*\.?\d*', text.strip())
    if match:
        try:
            return float(match.group().replace(",", ""))
        except Exception:
            return None
    return None

# Sanity test: generation works, injection changes output
t = TestResults()
sample_q = ds["test"][0]["question"]

gen_baseline = generate_answer(sample_q, inject_vector=None)
t.check(isinstance(gen_baseline, str) and len(gen_baseline) > 0, f"baseline generates text: '{gen_baseline[:60]}...'")

# Use DENORMALIZED flow states for injection (h_flow_denorm from cell 13)
gen_injected = generate_answer(sample_q, inject_vector=h_flow_denorm[0], alpha=0.3)
t.check(isinstance(gen_injected, str) and len(gen_injected) > 0, f"injected generates text: '{gen_injected[:60]}...'")

# Verify hook only fired once (prefill-only)
t.check(injection_state["fired"], "hook fired during generation")

parsed = parse_generated_answer("The answer is 42.")
t.check(parsed == 42.0, f"parse_generated_answer works: {parsed}")
parsed2 = parse_generated_answer("1,234 apples")
t.check(parsed2 == 1234.0, f"parse comma number: {parsed2}")

t.summary("Generation Helpers")

In [ ]:
# Cell: Phase 3b — Injection evaluation with layer sweep
# DEFINITIVE TEST: sweep layers × alphas for FlowThought, oracle, random, baseline

n_eval = min(200, test_data["h_cond"].shape[0])
test_questions = ds["test"].select(range(n_eval))
layers_to_test = [num_layers // 4, num_layers // 3, num_layers // 2]
alphas_to_test = [0.01, 0.05, 0.1, 0.3, 1.0]

print(f"Evaluating {n_eval} examples | layers: {layers_to_test} | alphas: {alphas_to_test}")
print()

# Helper: evaluate a method over n_eval examples
def run_eval(name, get_vector_fn, alpha, progress_every=50):
    results = []
    for i in range(n_eval):
        q = test_questions[i]["question"]
        true_ans = parse_answer(test_questions[i]["answer"])
        if true_ans is None:
            continue
        vec = get_vector_fn(i)
        gen = generate_answer(q, inject_vector=vec, alpha=alpha)
        pred = parse_generated_answer(gen)
        results.append({"true": true_ans, "pred": pred, "text": gen})
        if progress_every and (i + 1) % progress_every == 0:
            print(f"    {i+1}/{n_eval}")
    correct = sum(1 for r in results if r["pred"] is not None and abs(r["pred"] - r["true"]) < 0.5)
    return results, correct, len(results)

# --- Baseline (no injection) ---
print("Running baseline (no injection)...")
baseline_results = []
for i in range(n_eval):
    q = test_questions[i]["question"]
    true_ans = parse_answer(test_questions[i]["answer"])
    if true_ans is None:
        continue
    gen = generate_answer(q, inject_vector=None)
    pred = parse_generated_answer(gen)
    baseline_results.append({"true": true_ans, "pred": pred, "text": gen})
    if (i + 1) % 50 == 0:
        print(f"    {i+1}/{n_eval}")

baseline_correct = sum(1 for r in baseline_results if r["pred"] is not None and abs(r["pred"] - r["true"]) < 0.5)
baseline_total = len(baseline_results)
baseline_acc = baseline_correct / baseline_total
print(f"Baseline: {baseline_correct}/{baseline_total} = {baseline_acc:.4f}\n")

# --- Sweep layers for oracle, FlowThought (alpha=0.1), and random ---
all_layer_results = {}
for layer_idx in layers_to_test:
    set_injection_layer(layer_idx)
    print(f"=== Layer {layer_idx}/{num_layers} ===")

    # Oracle
    print(f"  Oracle (real CoT, alpha=0.1)...")
    oracle_res, oracle_c, oracle_t = run_eval(
        "oracle", lambda i: test_data["h_target"][i], alpha=0.1, progress_every=100)
    oracle_acc = oracle_c / oracle_t
    print(f"    Oracle: {oracle_c}/{oracle_t} = {oracle_acc:.4f}")

    # FlowThought at multiple alphas
    ft_alpha_results = {}
    for alpha in alphas_to_test:
        print(f"  FlowThought (alpha={alpha})...")
        ft_res, ft_c, ft_t = run_eval(
            f"ft_{alpha}", lambda i: h_flow_denorm[i], alpha=alpha, progress_every=100)
        ft_acc = ft_c / ft_t
        ft_alpha_results[alpha] = {"results": ft_res, "correct": ft_c, "total": ft_t, "acc": ft_acc}
        print(f"    FT(a={alpha}): {ft_c}/{ft_t} = {ft_acc:.4f}")

    # Random
    print(f"  Random (alpha=0.1)...")
    rand_res, rand_c, rand_t = run_eval(
        "random", lambda i: torch.randn(CONFIG["hidden_dim"]) * h_std + h_mean, alpha=0.1, progress_every=100)
    rand_acc = rand_c / rand_t
    print(f"    Random: {rand_c}/{rand_t} = {rand_acc:.4f}\n")

    all_layer_results[layer_idx] = {
        "oracle_acc": oracle_acc, "oracle_results": oracle_res,
        "ft_alpha_results": ft_alpha_results,
        "random_acc": rand_acc, "random_results": rand_res,
    }

# === Results Table ===
print("\n" + "=" * 80)
print("PHASE 3b: INJECTION RESULTS — LAYER × ALPHA SWEEP")
print("=" * 80)
print(f"\nBaseline (no injection): {baseline_acc:.4f}")
print()
for layer_idx in layers_to_test:
    lr = all_layer_results[layer_idx]
    print(f"--- Layer {layer_idx}/{num_layers} ---")
    print(f"  Oracle (real CoT):  {lr['oracle_acc']:.4f}")
    for alpha in alphas_to_test:
        fa = lr['ft_alpha_results'][alpha]
        print(f"  FT (alpha={alpha:<4}):   {fa['acc']:.4f}")
    print(f"  Random:             {lr['random_acc']:.4f}")
    print()

# === Verdict ===
best_oracle = max(lr["oracle_acc"] for lr in all_layer_results.values())
best_ft_overall = max(
    fa["acc"]
    for lr in all_layer_results.values()
    for fa in lr["ft_alpha_results"].values()
)
best_random = max(lr["random_acc"] for lr in all_layer_results.values())

# Find best layer+alpha combo
best_layer, best_alpha = None, None
for layer_idx in layers_to_test:
    for alpha in alphas_to_test:
        acc = all_layer_results[layer_idx]["ft_alpha_results"][alpha]["acc"]
        if acc == best_ft_overall:
            best_layer, best_alpha = layer_idx, alpha

print("=" * 80)
print("VERDICT")
print("=" * 80)
print(f"  Baseline:     {baseline_acc:.4f}")
print(f"  Best Oracle:  {best_oracle:.4f}")
print(f"  Best FT:      {best_ft_overall:.4f} (layer={best_layer}, alpha={best_alpha})")
print(f"  Best Random:  {best_random:.4f}")
print()

if best_oracle <= baseline_acc + 0.01:
    print("INJECTION MECHANISM BROKEN: Real CoT injection doesn't help at ANY layer.")
    print("The entire injection approach is flawed — not FM's fault.")
elif best_ft_overall > best_random + 0.02:
    print(f"SIGNAL DETECTED: FlowThought ({best_ft_overall:.4f}) > Random ({best_random:.4f})")
    print("FM learned useful structure. Worth scaling up.")
else:
    print(f"NO SIGNAL: FlowThought ({best_ft_overall:.4f}) ~ Random ({best_random:.4f})")
    if best_oracle > baseline_acc + 0.02:
        print("But oracle works — FM failed to learn the right structure.")
    print("FlowThought approach is DEAD for this architecture.")

# === Sample outputs (best layer) ===
best_lr = all_layer_results[best_layer]
best_ft_res = best_lr["ft_alpha_results"][best_alpha]["results"]
print(f"\n--- Sample outputs (layer={best_layer}, alpha={best_alpha}) ---")
for i in range(min(3, len(baseline_results))):
    print(f"\nExample {i+1} | True: {baseline_results[i]['true']}")
    print(f"  Baseline: {baseline_results[i]['text'][:80]}")
    print(f"  Oracle:   {best_lr['oracle_results'][i]['text'][:80]}")
    print(f"  FT:       {best_ft_res[i]['text'][:80]}")
    print(f"  Random:   {best_lr['random_results'][i]['text'][:80]}")

# === Tests ===
t = TestResults()
t.check(baseline_total > 0, f"evaluated {baseline_total} examples")
t.check(not torch.isnan(h_flow_denorm[:n_eval]).any().item(), "no NaN in FM states")
# Check injection changes outputs
n_changed = sum(1 for i in range(min(len(baseline_results), len(best_ft_res)))
                if baseline_results[i]["text"] != best_ft_res[i]["text"])
t.check(n_changed > 0, f"injection changes {n_changed}/{len(baseline_results)} outputs")
t.summary("Phase 3b: Injection Evaluation")

# Cleanup
if _current_hook_handle is not None:
    _current_hook_handle.remove()
del model, tokenizer
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
    print(f"\nLLM freed. GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell: Results summary — charts + JSON export

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

names = [r["name"] for r in results]
colors = ["#888", "#2ecc71", "#e74c3c", "#3498db"]

axes[0].bar(names, [r["correlation"] for r in results], color=colors)
axes[0].set_title("Probe: Correlation")
axes[0].set_ylabel("Pearson r")
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(names, [r["exact_match"] for r in results], color=colors)
axes[1].set_title("Probe: Exact Match")
axes[1].set_ylabel("Accuracy")
axes[1].tick_params(axis='x', rotation=15)

div_names = ["Real CoT", "FlowThought", "Random"]
div_vals = [div_real, div_flow, div_random]
axes[2].bar(div_names, div_vals, color=["#2ecc71", "#3498db", "#e74c3c"])
axes[2].set_title("Diversity (Cosine Distance)")
axes[2].set_ylabel("Mean Pairwise Distance")
axes[2].tick_params(axis='x', rotation=15)

# Phase 3b: best results per method
inj_names = ["Baseline", "Best\nOracle", "Best FT", "Best\nRandom"]
inj_vals = [baseline_acc, best_oracle, best_ft_overall, best_random]
inj_colors = ["#888", "#f39c12", "#3498db", "#e74c3c"]
axes[3].bar(inj_names, inj_vals, color=inj_colors)
axes[3].set_title("Injection Accuracy (best layer)")
axes[3].set_ylabel("Accuracy")
axes[3].tick_params(axis='x', rotation=15)

plt.suptitle(f"FlowThought PoC — {CONFIG['model_name'].split('/')[-1]} on GSM8K (A100)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Layer sweep heatmap
fig2, ax2 = plt.subplots(figsize=(8, 4))
layer_alpha_grid = np.array([
    [all_layer_results[l]["ft_alpha_results"][a]["acc"] for a in alphas_to_test]
    for l in layers_to_test
])
im = ax2.imshow(layer_alpha_grid, aspect='auto', cmap='viridis')
ax2.set_xticks(range(len(alphas_to_test)))
ax2.set_xticklabels([str(a) for a in alphas_to_test])
ax2.set_yticks(range(len(layers_to_test)))
ax2.set_yticklabels([f"Layer {l}" for l in layers_to_test])
ax2.set_xlabel("Alpha")
ax2.set_title("FlowThought Accuracy: Layer × Alpha")
for i in range(len(layers_to_test)):
    for j in range(len(alphas_to_test)):
        ax2.text(j, i, f"{layer_alpha_grid[i,j]:.3f}", ha="center", va="center",
                color="white" if layer_alpha_grid[i,j] < layer_alpha_grid.max()*0.7 else "black", fontsize=9)
plt.colorbar(im)
plt.tight_layout()
plt.show()

# Save results
summary = {
    "config": CONFIG,
    "phase3a_results": results,
    "phase3b_results": {
        "baseline_acc": baseline_acc,
        "best_oracle_acc": best_oracle,
        "best_ft_acc": best_ft_overall,
        "best_ft_layer": best_layer,
        "best_ft_alpha": best_alpha,
        "best_random_acc": best_random,
        "layer_sweep": {
            str(l): {
                "oracle_acc": all_layer_results[l]["oracle_acc"],
                "random_acc": all_layer_results[l]["random_acc"],
                "ft_alphas": {str(a): all_layer_results[l]["ft_alpha_results"][a]["acc"] for a in alphas_to_test},
            } for l in layers_to_test
        },
    },
    "diversity": {"real": div_real, "flow": div_flow, "random": div_random},
    "distribution_match": {"mean_dist": mean_dist, "std_dist": std_dist},
    "fm_final_loss": loss_history[-1],
    "probe_train_corr": train_corr,
}

with open("flowthought_results_r1_7b.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\nResults saved to flowthought_results_r1_7b.json")
print("\n" + "=" * 70)
print("FINAL INTERPRETATION")
print("=" * 70)
print(f"""
Model: {CONFIG['model_name']}
Data: {CONFIG['n_train']} train / {CONFIG['n_test']} test (full GSM8K)
FM: {CONFIG['fm_layers']} ResidualBlocks x {CONFIG['fm_hidden']} + AdaLN, {CONFIG['fm_epochs']} epochs

Phase 3a (probe):
  FlowThought corr: {r_flow['correlation']:.4f} | Real CoT: {r_real['correlation']:.4f} | Random: {r_random['correlation']:.4f}

Phase 3b (injection — the real test):
  Baseline:     {baseline_acc:.4f}
  Best Oracle:  {best_oracle:.4f}  (real CoT injection)
  Best FT:      {best_ft_overall:.4f}  (layer={best_layer}, alpha={best_alpha})
  Best Random:  {best_random:.4f}

Decision tree:
1. Oracle > Baseline? → injection mechanism works
2. FlowThought > Random? → FM learned useful structure
3. Both yes → scale up. Either no → kill the idea.
""")